In [ ]:
import pandas as pd
import numpy as np
import pygmt
from collections import Counter
import os
#import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist


In [ ]:
df_road = pd.read_csv("/home/velgueta/notebooks/project_Mt-Rainier_DAS/Text-files/Track_points.csv")
df_road.head()

In [ ]:
df_road.tail(10)

In [ ]:
# Assuming df_road is your DataFrame with "X" and "Y" columns representing longitude and latitude
df_road

# Start and end points
s_point = [46.771376, -121.786249]
e_point = [46.772903, -121.773978]


# Combine start and end points into a single array for efficient distance calculation
points = [[s_point[1], s_point[0]], [e_point[1], e_point[0]]]

# Calculate distances using Haversine formula
df_road['Distance_to_s'] = cdist(df_road[['X', 'Y']], [points[0]], metric='euclidean')
df_road['Distance_to_e'] = cdist(df_road[['X', 'Y']], [points[1]], metric='euclidean')

# Extract the part in-between
idx_min_s = df_road['Distance_to_s'].idxmin()
idx_min_e = df_road['Distance_to_e'].idxmin()
df_between = df_road.loc[min(idx_min_s, idx_min_e):max(idx_min_s, idx_min_e)]

# Print or use the results as needed
#print("Closest point to start:", closest_s_point)
#print("Closest point to end:", closest_e_point)
#print("Part in-between:")
print(df_between[['X', 'Y']])

In [ ]:
df_between.head(2)

In [ ]:
# list of seismic? stations in Mt Rainier
df_seis = pd.read_csv("/home/velgueta/notebooks/project_Mt-Rainier_DAS/Text-files/MtRainier_seis.txt",sep="|")
print(df_seis.head())
df_seis["Station"]

In [ ]:
# create short station list including stations which detect avalanche on 2023-12-10T12:09
sta_list = ['RER', 'RCS', 'RCM', 'LON', 'LO2', 'SIFT', 'RUSH', 'PR05', 'PR04', 'PARA', 'PANH', 'OBSR', 'ARAT']
df_seis_short = df_seis[df_seis["Station"].isin(sta_list)]
df_seis_short

In [ ]:
# loading swarm 

df_swarm = pd.read_csv("/home/velgueta/notebooks/project_Mt-Rainier_DAS/Text-files/pnsn_event_export_5dias.csv")
df_swarm.head()

In [ ]:
df_seismicity = pd.read_csv('/home/velgueta/notebooks/project_Mt-Rainier_DAS/Text-files/pnsn_event_export_20240613.csv')
df_seismicity


In [ ]:
df_template = pd.read_csv("/home/velgueta/notebooks/project_Mt-Rainier_DAS/Text-files/Template-location.csv")
df_template.head()

In [ ]:
import pygmt
import pandas as pd
import numpy as np
import os



fig = pygmt.Figure()
RR = [-122.1, -121.45, 46.65, 47.]
grid = pygmt.datasets.load_earth_relief(resolution="03s", region=RR)

#Create hillshade 
shading = pygmt.grdgradient(grid, azimuth = 315)#, direction = '45+o')

#fig.shift_origin(yshift="10c")

#Show grid and hillshade 
fig.grdimage(grid=grid, shading =shading, projection="M15c", frame="a", cmap="geo", transparency=30)
fig.colorbar(position='JMB+w9.+o3.c/1.c', frame='af+l"Elevation (m)"')

# Plot road for the DAS
fig.plot(x=df_road["X"], y=df_road["Y"], pen='3p,purple', label='DAS array')

# Plot stations
fig.plot(x=df_seis["Longitude"], y=df_seis["Latitude"],
         style="i0.50c", pen="1p,black", fill="black", transparency=30)

# Create a custom color palette (cpt) from orange to blue
pygmt.makecpt(cmap="viridis", series=[df_seismicity["Depth Km"].min(), df_seismicity["Depth Km"].max()], 
              continuous=True)#, reverse=True)

# Plot seismicity with color code by depth and size by magnitude
fig.plot(x=df_seismicity["Lon"], y=df_seismicity["Lat"],
         style="c", pen="1p,black", fill=df_seismicity["Depth Km"], 
         cmap=True, size=0.2 + df_seismicity["Magnitude"] * 0.1)

fig.plot(x=df_swarm["Lon"], y=df_swarm["Lat"],
         style="c", pen="1p,black", fill=df_swarm["Depth Km"], 
         cmap=True, size=0.2 + df_swarm["Magnitude"] * 0.1)

fig.plot(x=df_template["Lon"], y=df_template["Lat"],
         style="c", pen="1p,black", fill=df_template["Depth Km"], 
         cmap=True, size=0.2 + df_template["Magnitude"] * 0.1)

# Crear archivo temporal para la leyenda de tamaños de círculos
legend_file = "legend.txt"
with open(legend_file, "w") as f:
    f.write("S 0.1c c 0.4c white 1p,black 0.4c M1\n")
    f.write("S 0.1c c 0.6c white 1p,black 0.6c M3\n")
    f.write("S 0.1c c 0.8c white 1p,black 0.8c M5\n")

# Leyenda en la parte superior derecha
fig.plot(
    x=np.nan, y=np.nan, style="i0.6c", pen="1p,black", fill="black",
    label='PNSN+CVO network')
#fig.plot(
 #   x=np.nan, y=np.nan, style="c0.25c", pen="1p,black", fill="gray",
  #  label='Events since 8/25/2023')

# Ajustar el tamaño de la fuente solo para la leyenda en la parte superior derecha
with pygmt.config(FONT_ANNOT_PRIMARY="14p"):
    fig.legend(transparency=15, position="jTR+o0.2c")

# Agregar barra de color para la profundidad
fig.colorbar(position="JMR+w9c/0.5c+o1c/0c", frame='af+l"Depth (km)"')

# Leyenda para el tamaño de los círculos según la magnitud en la parte inferior izquierda
fig.legend(spec=legend_file, position="jBL+o0.5c/0.5c", box="+gwhite+p1p")

# Mostrar y guardar la figura
fig.show()
fig.savefig("./map_seis_das_swarm_2023-08-26_viridis.png", dpi=800)

# Eliminar el archivo temporal
os.remove(legend_file)


In [ ]:
import pygmt
import pandas as pd
import numpy as np
import os



fig = pygmt.Figure()
RR = [-122.1, -121.45, 46.65, 47.]
grid = pygmt.datasets.load_earth_relief(resolution="03s", region=RR)

#Create hillshade 
shading = pygmt.grdgradient(grid, azimuth = 315)#, direction = '45+o')

#fig.shift_origin(yshift="10c")

#Show grid and hillshade 
fig.grdimage(grid=grid, shading =shading, projection="M15c", frame="a", cmap="geo", transparency=30)
fig.colorbar(position='JMB+w9.+o3.c/1.c', frame='af+l"Elevation (m)"')

# Plot road for the DAS
fig.plot(x=df_road["X"], y=df_road["Y"], pen='3p,purple', label='DAS array')

# Plot stations
fig.plot(x=df_seis["Longitude"], y=df_seis["Latitude"],
         style="i0.50c", pen="1p,black", fill="black", transparency=30)

# Create a custom color palette (cpt) from orange to blue
pygmt.makecpt(cmap="turbo", series=[df_seismicity["Depth Km"].min(), df_seismicity["Depth Km"].max()], 
              continuous=True)#, reverse=True)

# Plot seismicity with color code by depth and size by magnitude
fig.plot(x=df_seismicity["Lon"], y=df_seismicity["Lat"],
         style="c", pen="1p,black", fill=df_seismicity["Depth Km"], 
         cmap=True, size=0.2 + df_seismicity["Magnitude"] * 0.1)

fig.plot(x=df_swarm["Lon"], y=df_swarm["Lat"],
         style="c", pen="1p,black", fill=df_swarm["Depth Km"], 
         cmap=True, size=0.2 + df_swarm["Magnitude"] * 0.1)

fig.plot(x=df_template["Lon"], y=df_template["Lat"],
         style="c", pen="1p,black", fill=df_template["Depth Km"], 
         cmap=True, size=0.2 + df_template["Magnitude"] * 0.1)

# Crear archivo temporal para la leyenda de tamaños de círculos
legend_file = "legend.txt"
with open(legend_file, "w") as f:
    f.write("S 0.1c c 0.4c white 1p,black 0.4c M1\n")
    f.write("S 0.1c c 0.6c white 1p,black 0.6c M3\n")
    f.write("S 0.1c c 0.8c white 1p,black 0.8c M5\n")

# Leyenda en la parte superior derecha
fig.plot(
    x=np.nan, y=np.nan, style="i0.6c", pen="1p,black", fill="black",
    label='PNSN+CVO network')
#fig.plot(
 #   x=np.nan, y=np.nan, style="c0.25c", pen="1p,black", fill="gray",
  #  label='Events since 8/25/2023')

# Ajustar el tamaño de la fuente solo para la leyenda en la parte superior derecha
with pygmt.config(FONT_ANNOT_PRIMARY="14p"):
    fig.legend(transparency=15, position="jTR+o0.2c")

# Agregar barra de color para la profundidad
fig.colorbar(position="JMR+w9c/0.5c+o1c/0c", frame='af+l"Depth (km)"')

# Leyenda para el tamaño de los círculos según la magnitud en la parte inferior izquierda
fig.legend(spec=legend_file, position="jBL+o0.5c/0.5c", box="+gwhite+p1p")

# Mostrar y guardar la figura
fig.show()
fig.savefig("./map_seis_das_swarm_2023-08-26_turbo.png", dpi=800)

# Eliminar el archivo temporal
os.remove(legend_file)
